# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maahilkureshi-design/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Main model: gradient boosting** (`HistGradientBoostingClassifier`) on the 18 past-only features. It fits because the data is tabular with very different scales, the relationships are not straight lines (past momentum has a U-shaped link to the label), and missing values are handled as they are. **Comparison model: regularized logistic regression**, a transparent linear model with median fill and a missing flag. **Baselines:** three transparent rules (relative momentum, raw momentum, impressions lost) and a shuffled-label null model.

All settings were fixed before any evaluation fold was scored. Nothing is tuned on the folds shown here.

In [ ]:
import os, json, shutil
import numpy as np
import pandas as pd
import duckdb
from sklearn.metrics import average_precision_score

# The saved snapshot files and shared outputs live in Google Drive (MyDrive/flyrank_capstone),
# so this notebook can run in its own session after capstone.ipynb has finished.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT, ON_COLAB = '/content/drive/MyDrive/flyrank_capstone', True
except ImportError:
    DRIVE_ROOT, ON_COLAB = '.', False
CACHE_DIR, EVAL_DIR, OUT_DIR = 'snap_cache', 'eval_cache', f'{DRIVE_ROOT}/outputs'
if ON_COLAB:
    assert os.path.exists(f'{DRIVE_ROOT}/snap_cache'), 'Not found in Google Drive: run capstone.ipynb first, then its last cell (copy to Drive).'
    shutil.copytree(f'{DRIVE_ROOT}/snap_cache', CACHE_DIR, dirs_exist_ok=True)      # local copy: faster than reading from Drive
    if os.path.exists(f'{DRIVE_ROOT}/eval_cache'):
        shutil.copytree(f'{DRIVE_ROOT}/eval_cache', EVAL_DIR, dirs_exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)
n_files = len([f for f in os.listdir(CACHE_DIR) if f.startswith('snap_') and f.endswith('.parquet')]) if os.path.exists(CACHE_DIR) else 0
assert n_files == 12, f'{n_files} snapshot files found, expected 12: run capstone.ipynb and its last cell (copy to Drive) first.'

LABEL_DAYS = 30
K_LIST = [20, 50]
MIN_CLIENT_PAGES_MODEL = 200
SPIKE_RULE = "imp_last30 >= 2 * GREATEST(imp_prev30, imp_prev60)"
UNIVERSE = f"n_elig >= {MIN_CLIENT_PAGES_MODEL} AND NOT ({SPIKE_RULE})"
PAST_COLS = ['imp_90', 'clicks_90', 'ctr_90', 'ctr_last30', 'pos_90', 'pos_last30', 'pos_prev30', 'pos_change',
             'imp_last30', 'imp_prev30', 'imp_prev60', 'active_days_90', 'cv_90',
             'log_mom', 'log_mom_rel', 'log_mom_prev', 'log_mom_prev_rel', 'page_age_days']
STATIC_COLS = ['word_count', 'char_count', 'content_type', 'main_intent', 'competition_level', 'competition',
               'search_volume', 'cpc', 'backlinks', 'category_count', 'keyword_token_count', 'url_char_count']
mcon = duckdb.connect()

def _clean(d):
    d['snap_t'] = pd.to_datetime(d['snap_t'])
    for c in PAST_COLS:
        if pd.api.types.is_timedelta64_dtype(d[c]):
            d[c] = d[c].dt.days
    return d

def load_train():
    cols = ', '.join(['snap_t', 'client_hash_id', 'y'] + PAST_COLS)
    return _clean(mcon.sql(f"SELECT {cols} FROM read_parquet('{CACHE_DIR}/snap_*.parquet') WHERE {UNIVERSE}").df())

def load_eval(name):
    path = f'{EVAL_DIR}/eval_{name}.parquet'
    assert os.path.exists(path), f'{path} is missing: run the build cells of capstone.ipynb first'
    cols = ', '.join(['snap_t', 'client_hash_id', 'y'] + PAST_COLS)
    return _clean(mcon.sql(f"SELECT {cols} FROM read_parquet('{path}') WHERE {UNIVERSE}").df())

def eval_rule(df, score_col, ks=K_LIST):
    rows = []
    for t, g in df.groupby('snap_t'):
        g = g.sort_values(score_col, ascending=False, kind='mergesort')
        base = g['y'].mean()
        row = {'snap_t': str(t)[:10], 'n': len(g), 'base_rate': round(base, 3)}
        for k in ks:
            p = g['y'].head(k).mean()
            row[f'p@{k}'] = round(p, 3)
            row[f'lift@{k}'] = round(p / base, 2) if base > 0 else np.nan
        ap = average_precision_score(g['y'], g[score_col]) if g['y'].sum() > 0 else np.nan
        row['ap'] = round(ap, 3)
        row['ap_lift'] = round(ap / base, 2) if base > 0 else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

LOG_COLS = ['imp_90', 'clicks_90', 'imp_last30', 'imp_prev30', 'imp_prev60']

def make_X(d, log=False):
    X = d[PAST_COLS].astype(float).copy()
    if log:
        for c in LOG_COLS:
            X[c] = np.log1p(X[c])
    return X

HGB_SETTINGS = dict(max_iter=200, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=200,
                    l2_regularization=1.0, early_stopping=False)
def make_hgb(seed=0):
    return HistGradientBoostingClassifier(random_state=seed, **HGB_SETTINGS)

def make_logit():
    return make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler(),
                         LogisticRegression(C=0.1, max_iter=1000))

def shuffle_within_snapshot(train, seed):
    y_new = train['y'].values.copy()
    rng = np.random.default_rng(seed)
    for _, idx in train.reset_index(drop=True).groupby('snap_t').indices.items():
        y_new[idx] = rng.permutation(y_new[idx])
    return y_new

data = load_train()
print('Gradient boosting settings:', HGB_SETTINGS)
print('Logistic regression: C=0.1, median fill plus missing flags, standardized, counts logged')
print(f"{len(data):,} labeled rows, {data['snap_t'].nunique()} snapshots")

Mounted at /content/drive
Gradient boosting settings: {'max_iter': 200, 'learning_rate': 0.05, 'max_leaf_nodes': 15, 'min_samples_leaf': 200, 'l2_regularization': 1.0, 'early_stopping': False}
Logistic regression: C=0.1, median fill plus missing flags, standardized, counts logged
219,258 labeled rows, 12 snapshots


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Time-aware walk-forward with a 30-day embargo.** The model is scored on each of the last six training snapshots in turn, and is trained only on earlier snapshots. A training snapshot is dropped if its 30-day label window would run past the evaluation date. A random row split is not used for any claim: the same page appears in neighbouring months, so a random split lets a page's earlier months leak into its test rows (`w06_validation_audit` shows the size of that effect). A client-held-out check and the validation and test months are reported in the capstone.

In [ ]:
EVAL_SNAPS = sorted(data['snap_t'].dt.strftime('%Y-%m-%d').unique())[-6:]
fold_rows = []
for t_eval in EVAL_SNAPS:
    te_t = pd.Timestamp(t_eval)
    train = data[data['snap_t'] + pd.Timedelta(days=LABEL_DAYS) <= te_t]
    test = data[data['snap_t'] == te_t]
    ok = bool(len(train)) and (train['snap_t'].max() + pd.Timedelta(days=LABEL_DAYS) <= te_t)
    fold_rows.append({'evaluate_on': t_eval, 'train_rows': len(train), 'latest_training_snapshot': str(train['snap_t'].max())[:10],
                      'test_rows': len(test), 'embargo_respected': ok})
print(pd.DataFrame(fold_rows).to_string(index=False))
assert all(r['embargo_respected'] for r in fold_rows if r['train_rows'])

evaluate_on  train_rows latest_training_snapshot  test_rows  embargo_respected
 2025-10-31       35090               2025-09-30      13547               True
 2025-11-30       48637               2025-10-31      17738               True
 2025-12-31       66375               2025-11-30      27117               True
 2026-01-31       93492               2025-12-31      30498               True
 2026-02-28       93492               2025-12-31      41468               True
 2026-03-31      165458               2026-02-28      53800               True


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same data, same folds, same metrics as the baselines. A random queue has lift 1.0 by construction; the null model is the main model trained on shuffled labels. Lift is precision (or average precision) divided by the base rate of that snapshot, so months with different base rates can be compared. In the capstone run, over these six folds, the main model's mean AP lift was about 2.3, against about 1.3 for the best rule and about 1.1 for the null model.

In [ ]:
parts = []
for t_eval in EVAL_SNAPS:
    te_t = pd.Timestamp(t_eval)
    train = data[data['snap_t'] + pd.Timedelta(days=LABEL_DAYS) <= te_t].reset_index(drop=True)
    test = data[data['snap_t'] == te_t].copy()
    if len(train) == 0 or len(test) == 0 or train['y'].nunique() < 2:
        print(t_eval, 'skipped (no usable training rows)')
        continue
    test['hgb'] = make_hgb(0).fit(make_X(train), train['y'].values).predict_proba(make_X(test))[:, 1]
    test['logit'] = make_logit().fit(make_X(train, log=True), train['y'].values).predict_proba(make_X(test, log=True))[:, 1]
    test['null'] = make_hgb(0).fit(make_X(train), shuffle_within_snapshot(train, 0)).predict_proba(make_X(test))[:, 1]
    test['rule_rel_momentum'] = -test['log_mom_rel']
    test['rule_raw_momentum'] = -test['log_mom']
    test['rule_impressions_lost'] = test['imp_prev30'] - test['imp_last30']
    parts.append(test)
res = pd.concat(parts)

SCORES = ['hgb', 'logit', 'null', 'rule_rel_momentum', 'rule_raw_momentum', 'rule_impressions_lost']
summary = []
for s in SCORES:
    e = eval_rule(res, s)
    summary.append({'score': s, **e[['p@20', 'lift@20', 'p@50', 'lift@50', 'ap_lift']].mean().round(3).to_dict()})
summary = pd.DataFrame(summary)
print(summary.to_string(index=False))
summary.to_csv(os.path.join(OUT_DIR, 'walkforward_summary.csv'), index=False)

print('\nMain model, per snapshot:')
print(eval_rule(res, 'hgb').to_string(index=False))

                score  p@20  lift@20  p@50  lift@50  ap_lift
                  hgb 0.508    7.633 0.423    6.255    2.282
                logit 0.450    7.003 0.447    6.758    1.962
                 null 0.275    3.267 0.237    2.830    1.138
    rule_rel_momentum 0.250    4.028 0.237    3.882    1.337
    rule_raw_momentum 0.233    3.285 0.220    3.430    1.342
rule_impressions_lost 0.258    4.217 0.243    3.897    1.288

Main model, per snapshot:
    snap_t     n  base_rate  p@20  lift@20  p@50  lift@50    ap  ap_lift
2025-10-31 13547      0.048  0.40     8.38  0.26     5.44 0.118     2.47
2025-11-30 17738      0.040  0.05     1.27  0.12     3.04 0.092     2.32
2025-12-31 27117      0.073  0.85    11.64  0.64     8.77 0.189     2.59
2026-01-31 30498      0.047  0.70    14.93  0.54    11.52 0.112     2.40
2026-02-28 41468      0.080  0.50     6.29  0.44     5.53 0.178     2.23
2026-03-31 53800      0.167  0.55     3.29  0.54     3.23 0.280     1.68


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Where is the model wrong, and what does it lean on? Three views on the last fold: permutation importance (how much average precision drops when one feature is shuffled), the profile of the model's top 20 pages against all pages, and the positives the model ranks in the bottom half. In the capstone run the model leaned mostly on volatility (`cv_90`) and the number of active days; its top picks were low-volume, intermittently visible pages at deep positions that were already falling. That makes them hard cases for a reviewer, because small counts are noisy, and it is why the validation audit also reports pages with at least 1,000 impressions.

In [ ]:
from sklearn.inspection import permutation_importance

t_eval = pd.Timestamp(EVAL_SNAPS[-1])
train = data[data['snap_t'] + pd.Timedelta(days=LABEL_DAYS) <= t_eval].reset_index(drop=True)
test = data[data['snap_t'] == t_eval].copy()
model = make_hgb(0).fit(make_X(train), train['y'].values)
X_te = make_X(test)
test['score'] = model.predict_proba(X_te)[:, 1]

imp = permutation_importance(model, X_te, test['y'].values, scoring='average_precision',
                             n_repeats=3, random_state=0, n_jobs=1)
importance = pd.Series(imp.importances_mean, index=PAST_COLS).sort_values(ascending=False)
print(f'Permutation importance, fold {EVAL_SNAPS[-1]} (drop in average precision):')
print(importance.head(8).round(4).to_string())
importance.round(5).to_csv(os.path.join(OUT_DIR, 'permutation_importance.csv'), header=['importance'])

PROFILE = ['imp_last30', 'imp_prev30', 'active_days_90', 'cv_90', 'log_mom_rel', 'pos_last30', 'page_age_days']
top = test.nlargest(20, 'score')
prof = pd.DataFrame({'top20_median': top[PROFILE].median(), 'all_pages_median': test[PROFILE].median()}).round(2)
print('\nTop 20 pages versus all pages (medians):')
print(prof.to_string())

# The positives the model ranks in the bottom half
test['rank_pct'] = test['score'].rank(ascending=False, pct=True)
missed = test[(test['y'] == 1) & (test['rank_pct'] > 0.5)]
caught = test[(test['y'] == 1) & (test['rank_pct'] <= 0.1)]
print(f"\nPositives ranked in the bottom half: {len(missed):,} of {int(test['y'].sum()):,}; positives in the top 10%: {len(caught):,}")
cmp = pd.DataFrame({'missed_median': missed[PROFILE].median(), 'caught_in_top10pct_median': caught[PROFILE].median()}).round(2)
print(cmp.to_string())

# Per client on this fold (client size rank only, no identifiers)
rows = []
sizes = test.groupby('client_hash_id').size().sort_values(ascending=False)
for rank, (c, n) in enumerate(sizes.items(), 1):
    g = test[test['client_hash_id'] == c]
    if g['y'].sum() and g['y'].mean() > 0:
        rows.append({'client_rank': rank, 'n': n, 'base_rate': round(g['y'].mean(), 3),
                     'ap_lift': round(average_precision_score(g['y'], g['score']) / g['y'].mean(), 2)})
print('\nPer client on this fold:')
print(pd.DataFrame(rows).to_string(index=False))

Permutation importance, fold 2026-03-31 (drop in average precision):
cv_90               0.0938
active_days_90      0.0226
page_age_days       0.0135
ctr_last30          0.0115
log_mom_prev_rel    0.0095
pos_last30          0.0092
log_mom_prev        0.0091
log_mom             0.0084

Top 20 pages versus all pages (medians):
                top20_median  all_pages_median
imp_last30            209.00            918.00
imp_prev30            264.50            936.00
active_days_90         72.00             90.00
cv_90                   1.00              0.59
log_mom_rel            -0.65             -0.07
pos_last30             59.80              8.37
page_age_days         216.50            248.00

Positives ranked in the bottom half: 2,790 of 8,987; positives in the top 10%: 1,848
                missed_median  caught_in_top10pct_median
imp_last30             635.00                     454.00
imp_prev30             595.00                     481.50
active_days_90          90.00           

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.